# Assignment 02: Polars on EHR Event Logs

Build a Polars pipeline that summarizes diagnosis prevalence from synthetic EHR events. Use lazy scans, filtering, joins, and group-bys to compute site-level diabetes prevalence.

## Setup


In [2]:
import polars as pl
import yaml
from pathlib import Path
from datetime import datetime
from generate_test_data import generate_test_data

print(f"Polars version: {pl.__version__}")
print("Environment ready!")

Polars version: 1.37.1
Environment ready!


## Configuration


In [3]:
with open("config.yaml") as f:
    config = yaml.safe_load(f)

print("Config loaded:")
print(f"  Patients: {config['data']['patients_path']}")
print(f"  Sites: {config['data']['sites_path']}")
print(f"  Events: {config['data']['events_path']}")
print(f"  ICD-10 lookup: {config['data']['icd10_path']}")

Config loaded:
  Patients: data/patients.parquet
  Sites: data/sites.parquet
  Events: data/events.parquet
  ICD-10 lookup: data/icd10_codes.parquet


## Generate data


In [19]:
SIZE = config["data"]["size"]
DATA_DIR = Path(config["data"]["dir"])

# Create output directory if it doesn't exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Generate data - "medium" takes ~10 seconds on my laptop
generate_test_data(size=SIZE, output_dir=DATA_DIR)

INFO Loading codebooks
INFO Generating sites
INFO Generating patients
INFO Generating events
INFO Wrote 100000 patients
INFO Wrote 8 sites
INFO Wrote 7487826 events
INFO Output directory: data


## Hints (optional)

- Distinct patient counts: call `.unique()` before `group_by()`.
  - Example: `events.select(["site_id", "patient_id"]).unique()`
- Prefix filter for ICD-10: `pl.col("code").str.starts_with(prefix)`
- Optional polish: `.fill_null(0)` after a left join, and `.round(3)` on prevalence

## Part 1: Lazy Data Loading

Use `pl.scan_parquet()` to create LazyFrames without loading data into memory.


In [5]:
# TODO: Scan patients, sites, events, and ICD-10 lookup
patients = pl.scan_parquet(config['data']['patients_path'])
sites = pl.scan_parquet(config['data']['sites_path'])
events = pl.scan_parquet(config['data']['events_path'])
icd10 = pl.scan_parquet(config['data']['icd10_path'])

# Check schemas (fast, still lazy)
if patients is not None:
    print("Patients schema:")
    print(patients.collect_schema())

if events is not None:
    print("Events schema:")
    print(events.collect_schema())

print(events.select('event_ts').head().collect())


Patients schema:
Schema({'patient_id': String, 'dob': String, 'gender': String, 'zip_code': String, 'home_site_id': String})
Events schema:
Schema({'event_id': String, 'patient_id': String, 'site_id': String, 'event_ts': String, 'record_type': String, 'code': String})
shape: (5, 1)
┌─────────────────────┐
│ event_ts            │
│ ---                 │
│ str                 │
╞═════════════════════╡
│ 2023-02-09T21:08:40 │
│ 2023-03-03T07:29:58 │
│ 2023-08-23T05:52:43 │
│ 2024-12-01T09:40:39 │
│ 2024-02-15T07:29:46 │
└─────────────────────┘


## Part 2: Filter and Prep Events

Filter to the assignment date window and extract ICD-10 diagnosis events.


In [6]:
start_date = datetime.fromisoformat(config["data"]["start_date"])

# TODO: parse event_ts to Datetime
events = events.with_columns(pl.col('event_ts').str.strptime(pl.Datetime))
# TODO: filter events to event_ts >= start_date
events_filtered = events.with_columns(pl.col('events_ts')>= start_date)
# TODO: filter to record_type == "ICD-10-CM" for diagnosis events
dx_events = events.with_columns(pl.col('record_type')=="ICD-10-CM")

## Part 3: Diagnosis Prevalence by Site

Compute the percent of patients per site with a type 2 diabetes diagnosis.


In [18]:
prefix = config["data"]["diabetes_prefix"]

# TODO: Filter dx_events to ICD-10 codes starting with prefix
# TODO: total patients per site (unique patient_id from events_filtered)
# TODO: diabetes patients per site (unique patient_id from filtered dx)
# TODO: join counts + site names, calculate prevalence

dx_summary = dx_events.filter(pl.col('code').str.starts_with(prefix))

if dx_summary is not None:
    print(dx_summary.collect_schema())

total_patients_persite = (events_filtered.select(['site_id','patient_id']).unique()
                          .group_by('site_id')
                          .agg(pl.len().alias('total_patients')))
total_db_patients_persite = (dx_summary.select(['site_id','patient_id']).unique()
                          .group_by('site_id')
                          .agg(pl.len().alias('total_patients_db')))
prevalence = (
    total_patients_persite
    .join(total_db_patients_persite, on='site_id', how='left')
    .with_columns(
        (pl.col('total_patients_db') / pl.col('total_patients'))
        .round(3)
        .alias('prevalence')
    )
    .fill_null(0))

Schema({'event_id': String, 'patient_id': String, 'site_id': String, 'event_ts': Datetime(time_unit='us', time_zone=None), 'record_type': Boolean, 'code': String})


## Part 4: Collect and Export


In [20]:
# TODO: collect dx_summary using streaming engine
# TODO: create outputs directory
# TODO: write Parquet + CSV outputs using config paths
dx_summary = dx_summary.collect(engine="streaming")
OUTPUT_DIR = Path(config["outputs"]["dx_summary_parquet"]).parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
dx_summary.write_parquet(config["outputs"]["dx_summary_parquet"])
dx_summary.write_csv(config["outputs"]["dx_summary_csv"])

if dx_summary is not None:
    print("Outputs ready")




Outputs ready


## Validation


In [21]:
outputs = [
    config["outputs"]["dx_summary_parquet"],
    config["outputs"]["dx_summary_csv"],
]

missing = [path for path in outputs if not Path(path).exists()]
if missing:
    print("Missing outputs:", missing)
else:
    print("All outputs created")

All outputs created


## Next Steps (Optional)

1. Run `python -m pytest .github/tests/test_assignment.py -v` in your terminal.
2. Use exploratory data analysis (EDA) or visualization techniques to get a feel for the dataset
